<a href="https://colab.research.google.com/github/touda/test/blob/rainfall-correlation/RBFN_SIMPLE.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# === ハイブリッドRBFN構築コード（格子RBF + 学習済クラスタRBF + 入力・中心点可視化付き）===
import pandas as pd
import numpy as np
from sklearn.cluster import KMeans
import os

# Google Driveと接続を行います。これを行うことで、Driveにあるデータにアクセスできるようになります。
# 下記セルを実行すると、Googleアカウントのログインを求められますのでログインしてください。
from google.colab import drive
drive.mount('/content/drive')
os.chdir('/content/drive/MyDrive/Colab Notebooks')

# === データ読み込み ===
df = pd.read_csv("rainfall_2009_2023_no_year.csv")
df.columns = ['rainfall', 'soil']
df['rainfall'] = pd.to_numeric(df['rainfall'], errors='coerce')
df['soil'] = pd.to_numeric(df['soil'], errors='coerce')
df_clean = df.dropna()

# === パラメータ設定 ===
delta_x, delta_y = 5, 1
sigma_x, sigma_y = 145, 80
lambda_min, lambda_max = 1, 10000

# === 格子点とクラスタ代表点抽出 ===
x_range = np.arange(0, 305, delta_x)
y_range = np.arange(0, 141, delta_y)
grid_centers, grid_counts = [], []
cluster_centers, cluster_lambdas = [], []

for cx in x_range:
    for cy in y_range:
        mask = (
            (df_clean['soil'] >= cx) & (df_clean['soil'] < cx + delta_x) &
            (df_clean['rainfall'] >= cy) & (df_clean['rainfall'] < cy + delta_y)
        )
        local_points = df_clean.loc[mask, ['soil', 'rainfall']].values
        count = len(local_points)
        if count > 0:
            # 格子中心
            grid_centers.append((cx, cy))
            grid_counts.append(count)
            # クラスタ重心（soil: X軸, rainfall: Y軸）
            kmeans = KMeans(n_clusters=1, random_state=0, n_init=10).fit(local_points)
            soil_center, rainfall_center = kmeans.cluster_centers_[0]
            cluster_centers.append((soil_center, rainfall_center))
            lam = lambda_max - (lambda_max - lambda_min) / (1 + count) #(7)式
            cluster_lambdas.append(lam)

# === RBF関数定義 ===
def rbf(x, y, center, sx, sy):
    return np.exp(-(((x - center[0])**2) / (2 * sx**2) + ((y - center[1])**2) / (2 * sy**2))) # (4)式

# === 学習：H行列と重みw ===
m = len(cluster_centers)
H = np.zeros((m, m))
for i, xi in enumerate(cluster_centers):
    for j, cj in enumerate(cluster_centers):
        H[i, j] = rbf(xi[0], xi[1], cj, sigma_x, sigma_y) #(5)式

Lambda = np.diag(cluster_lambdas)
y_teacher = np.ones(m)
w = np.linalg.inv(H.T @ H + Lambda) @ H.T @ y_teacher #(6)式
# === 応答面Z生成 ===
x_vals = np.arange(0, 296, 5)
y_vals = np.arange(0, 141, 1)
X, Y = np.meshgrid(x_vals, y_vals)
Z = np.zeros_like(X, dtype=float)

# 格子RBF（信頼度 λ を乗算）
for (cx, cy), lam in zip(grid_centers, cluster_lambdas):
    Z += lam * rbf(X, Y, (cx, cy), sigma_x, sigma_y)

# 学習済クラスタRBF（重み付き合成）
for j, cj in enumerate(cluster_centers):
    Z += w[j] * rbf(X, Y, cj, sigma_x, sigma_y)

#正規化調整
# 元のZの最小値・最大値
Z_min, Z_max = Z.min(), Z.max()

# 任意の範囲に変換したい（例：a〜b）
a = 0.0
b = 1.1
Z_norm = a + (b - a) * (Z - Z_min) / (Z_max - Z_min)

# === CSV保存 ===
df_csv = pd.DataFrame(np.round(Z_norm, 5), index=y_vals, columns=x_vals)
df_csv.to_csv("rbfn_combined_output2.csv", float_format="%.5f")

Mounted at /content/drive
